<a href="https://colab.research.google.com/github/dabs254/flyrank-ml/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dabs254/flyrank-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**My lane: Lane 2 — Refresh / Content Opportunity Scoring.**

The question behind this lane is simple: *which pages should an editor review and update first?* Websites have far more pages than anyone has time to fix, so the real value is in picking the right ones.

I chose this lane for three reasons. First, in Week 1 I ran the starter pipeline and saw a learned model pick declining pages much better than a hand-written rule (Precision@50 of about 0.74 vs 0.24), so I know there is real signal to work with. Second, my own Week 1 discovery — newer pages were declining more often than older ones — goes against a common assumption, which tells me a simple "fix the oldest pages" rule would miss a lot. Third, the result is practical: a ranked list with reasons that a real person could act on.

In [5]:
# Setup: get the repo files (Colab) and load the starter dataset
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("data/raw"):
    if not os.path.isdir("flyrank-ml"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dabs254/flyrank-ml"], check=True)
    os.chdir("flyrank-ml")
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages from {df['client_id'].nunique()} clients, {df.shape[1]} columns")

30,000 pages from 32 clients, 44 columns


## 2. The question: decision, action, cost of a wrong call

**My question:** *Out of all the pages on a site, which ones should a content editor review first because they are most likely to be losing search traffic?*

- **Unit of analysis:** one page (one row in the data).
- **Output:** a ranked list of pages — a "review queue" — with a short reason for each page (for example: "high impressions, dropping, not updated in a long time") and a suggested action.
- **Decision it improves:** where a content team spends its limited editing time each week.
- **Who acts on it:** a content editor or SEO specialist. They take the top pages from the list and decide whether to refresh the content, expand it, fix the title, merge it with a similar page, or just keep an eye on it.
- **Cost of a wrong call:**
  - If the list puts a healthy page near the top, an editor wastes hours on a page that didn't need help.
  - If the list misses a page that really is declining, that page keeps losing traffic while no one notices.
  - Because editor time is limited, the second mistake usually matters more for high-traffic pages — so the list should pay extra attention to pages with lots of visibility.

**Why data / ML can help:** a simple rule like "fix the oldest pages" or "fix pages not updated in 6 months" is easy to write, but my Week 1 work showed it only catches a small share of declining pages. Whether a page is declining depends on many signals at once (age, position, clicks, impressions, engagement), and those patterns are too tangled to write by hand. A model can learn from many examples — but I will always compare it to a simple rule first, so it has to prove it is actually better.

**How I'll measure success:** Precision@K — out of the top K pages on my list (for example the top 50), how many are really declining. This matches how the list is used: an editor only ever looks at the top of it.

In [6]:
# How big is the problem? Why a ranked list is needed
declining = df["trend_direction"].str.lower().eq("down")
print(f"Declining pages: {declining.sum():,} of {len(df):,} ({declining.mean():.0%})")
print("No team can review this many pages at once -> they need to know which ones come FIRST.")

Declining pages: 16,262 of 30,000 (54%)
No team can review this many pages at once -> they need to know which ones come FIRST.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter dataset (30,000 anonymized pages, 32 clients) that make this lane worth pursuing:

1. **9,961 declining pages are still highly visible** (500+ impressions in 90 days). These pages still matter to searchers — so they are worth saving, but there are far too many to review one by one.
2. **Declining pages carry about half of all search visibility** — roughly 51% of all impressions and 45% of all clicks. This isn't a problem in a forgotten corner of the site; it affects a big part of the traffic.
3. **Only 174 pages (0.6%) haven't been updated in 180+ days.** So a simple "fix the stale pages" rule could reach at most 174 of the 16,262 declining pages. Most declining pages are *not* old or stale — which matches my Week 1 discovery and shows why a smarter ranking is needed.

In [7]:
# Three numbers that support Lane 2
declining = df["trend_direction"].str.lower().eq("down")
visible = df["impressions_90d"] >= 500
print(f"1) Declining pages that are still visible (500+ impressions in 90 days): {(declining & visible).sum():,}")

imp_share = df.loc[declining, "impressions_90d"].sum() / df["impressions_90d"].sum()
clk_share = df.loc[declining, "clicks_90d"].sum() / df["clicks_90d"].sum()
print(f"2) Share of all impressions on declining pages: {imp_share:.0%}   |   share of all clicks: {clk_share:.0%}")

stale = df["days_since_last_update"] >= 180
print(f"3) Pages not updated in 180+ days: {stale.sum():,} ({stale.mean():.1%} of all pages)")
print(f"   -> a 'fix stale pages' rule could reach at most {stale.sum():,} of the {declining.sum():,} declining pages")

1) Declining pages that are still visible (500+ impressions in 90 days): 9,961
2) Share of all impressions on declining pages: 51%   |   share of all clicks: 45%
3) Pages not updated in 180+ days: 174 (0.6% of all pages)
   -> a 'fix stale pages' rule could reach at most 174 of the 16,262 declining pages


## 4. Careful words: what I can and can't claim

**What my work WILL be able to say:**
- *Observed* patterns in this data — for example, "in this sample, pages with these signals were more often declining."
- *Measured* results — for example, "my ranking's Precision@50 was X, compared with Y for a simple rule, on clients the model never saw during training."
- *Directional* findings — signs pointing one way, not final proof.
- *Decision support* — a ranked list that helps an editor decide where to look first. The editor still makes the final call.

**What my work will NEVER claim:**
- That I "predicted Google's algorithm" or found how Google ranks pages.
- That updating a page *will* make it recover. My list says "review this page first," not "fixing this page guarantees more traffic." Proving that would need a real experiment.
- That something *causes* decline. I can only show what tends to go together.
- Anything about specific clients or websites — the data is anonymized and stays that way.

**One honest limit to remember:** "declining" in this dataset comes from the `trend_direction` column. Since that column (and `trend_pct`, which it is calculated from) *is* the answer, I will never use them as inputs to a model — that would be leakage, like giving a student the answer key.

In [8]:
# Leakage guard: these columns ARE the answer, so they must never be model inputs
label_cols = ["trend_direction", "trend_pct"]
id_cols = ["content_id", "client_id"]          # IDs are for grouping/splitting only, never features
safe_features = [c for c in df.columns if c not in label_cols + id_cols]
print(f"Kept out of the model: {label_cols + id_cols}")
print(f"Columns still available as possible features: {len(safe_features)}")

Kept out of the model: ['trend_direction', 'trend_pct', 'content_id', 'client_id']
Columns still available as possible features: 40


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.